# Assignment 2: Strands Travel Planner
Build an AI agent using Strands Agents that creates a one-day travel plan for a city.

The agent must:

Use a weather tool to get the current weather.
Use a web/search tool to find 3 popular attractions.
Use a calculator tool to estimate the total cost of visiting the attractions.
Combine the results into a concise one-day itinerary.

```text
USER
  │
  ▼
┌─────────────────────┐
│    STRANDS AGENT    │
│                     │
│  Tool orchestration │
└──────────┬──────────┘
           │
           ▼
     WEATHER TOOL
           │
           ▼
         PyOWM
           │
           ▼
    OpenWeather API
           │
           ▼
      SEARCH TOOL
           │
           ▼
    CALCULATOR TOOL
           │
           ▼
   ┌─────────────────┐
   │ FINAL ITINERARY │
   └─────────────────┘

In [1]:
%pip install -q strands-agents tavily-python openai pyowm python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import logging
import os
import requests

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.executors import SequentialToolExecutor
from tavily import TavilyClient

In [3]:
# Load API keys from the local .env file
load_dotenv()

required_keys = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "OPENWEATHER_API_KEY": os.getenv("OPENWEATHER_API_KEY"),
    "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")
}

missing_keys = [
    name for name, value in required_keys.items()
    if not value or value.startswith("your_")
]

if missing_keys:
    raise ValueError(
        "Missing API key(s): " + ", ".join(missing_keys) +
        ". Add real values to the workspace .env file."
    )

GROQ_API_KEY = required_keys["GROQ_API_KEY"]
OPENWEATHER_API_KEY = required_keys["OPENWEATHER_API_KEY"]
TAVILY_API_KEY = required_keys["TAVILY_API_KEY"]

print("API credentials loaded from .env.")

API credentials loaded from .env.


In [4]:
# Suppress non-critical Strands OpenAI provider warnings
logging.getLogger("strands.models.openai").setLevel(logging.ERROR)

In [5]:
#Model and Search Client
groq_model = OpenAIModel(
    model_id="openai/gpt-oss-120b",
    client_args={
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1"
    }
)

tavily_client = TavilyClient(
    api_key=TAVILY_API_KEY
)

print("Groq model and Tavily client initialized.")

Groq model and Tavily client initialized.


In [6]:
#Weather Tool
from pyowm import OWM

owm = OWM(OPENWEATHER_API_KEY)
weather_manager = owm.weather_manager()


@tool
def get_current_weather(city: str) -> dict:
    """Get the current weather for a city using PyOWM.

    Args:
        city: Name of the city, such as Kathmandu or Pokhara.
    """

    try:
        observation = weather_manager.weather_at_place(city)
        weather = observation.weather
        location = observation.location

        return {
            "success": True,
            "city": location.name,
            "country": location.country,
            "temperature_c": round(weather.temperature("celsius")["temp"], 1),
            "feels_like_c": round(
                weather.temperature("celsius")["feels_like"], 1
            ),
            "condition": weather.detailed_status,
            "humidity_percent": weather.humidity,
            "wind_speed_mps": weather.wind()["speed"]
        }

    except Exception as error:
        return {
            "success": False,
            "error": f"Weather lookup failed: {error}"
        }

In [7]:
#Search Tool
@tool
def search_attractions(city: str) -> dict:
    """Search the web for popular tourist attractions and
    entrance-fee information for a city.

    Args:
        city: Name of the city.
    """

    query = (
        f"popular tourist attractions in {city} "
        f"top attractions entrance fee ticket price"
    )

    try:
        response = tavily_client.search(
            query=query,
            search_depth="advanced",
            max_results=6
        )

    except Exception as error:
        return {
            "success": False,
            "error": f"Search failed: {error}"
        }

    results = response.get("results", [])

    sources = [
        {
            "title": result.get("title", ""),
            "content": result.get("content", ""),
            "url": result.get("url", "")
        }
        for result in results
    ]

    return {
        "success": True,
        "city": city,
        "sources": sources
    }

In [8]:
#Calculator Tool
@tool
def calculate_total_cost(
    attraction_1_cost: float,
    attraction_2_cost: float,
    attraction_3_cost: float
) -> dict:
    """Calculate the combined entrance cost of three attractions.

    Args:
        attraction_1_cost: Entrance cost of attraction 1.
        attraction_2_cost: Entrance cost of attraction 2.
        attraction_3_cost: Entrance cost of attraction 3.
    """

    total = (
        attraction_1_cost
        + attraction_2_cost
        + attraction_3_cost
    )

    return {
        "attraction_1_cost": attraction_1_cost,
        "attraction_2_cost": attraction_2_cost,
        "attraction_3_cost": attraction_3_cost,
        "total_cost": round(total, 2)
    }

In [9]:
#Strands Agent
travel_agent = Agent(
    model=groq_model,

    tools=[
        get_current_weather,
        search_attractions,
        calculate_total_cost
    ],

    tool_executor=SequentialToolExecutor(),

    # Prevent intermediate streaming output
    callback_handler=None,

    system_prompt="""
You are a reliable one-day travel planning agent.

Create a practical one-day travel itinerary for the city provided
by the user.

FOLLOW THIS TOOL WORKFLOW:

1. Call get_current_weather(city).
2. Call search_attractions(city).
3. Select exactly THREE popular attractions from the search results.
4. Determine the entrance cost for each selected attraction using
   the available search evidence.
5. Call calculate_total_cost() with the three costs.
6. Generate the final itinerary using the collected information.

RULES:

- Always use the weather tool.
- Always use the search tool.
- Always use the calculator tool.
- Never calculate the final total yourself.
- Use one consistent currency for all three attraction costs.
- If a fee is not clearly available, identify it as an estimate.
- Do not invent weather information.
- Do not present estimated prices as official prices.
- Select exactly three attractions.

FINAL RESPONSE:

CURRENT WEATHER

City:
Temperature:
Condition:

ONE-DAY ITINERARY

Morning:
...

Afternoon:
...

Evening:
...

ESTIMATED ATTRACTION COST

1. Attraction — Cost
2. Attraction — Cost
3. Attraction — Cost

Total estimated cost: ...

TRAVEL TIP

...

Return only the final travel plan.

Do not include:
- internal reasoning
- tool-selection reasoning
- tool execution details
- intermediate calculations
"""
)

print("Travel Planner Agent initialized.")

Travel Planner Agent initialized.


In [10]:
#User Input for city
city = input("Enter a city: ").strip()

if not city:
    raise ValueError("Please enter a city name.")

print(f"Destination: {city}")

Destination: boston


In [11]:
#Execute Agent
prompt = f"""
Create a one-day travel plan for {city}.

Use the required weather, web search, and calculator tools.
Find exactly three popular attractions and calculate their combined
estimated entrance cost.

Return only the final travel plan.
"""

result = travel_agent(prompt)

print("\n" + "=" * 70)
print(f"ONE-DAY TRAVEL PLAN — {city.upper()}")
print("=" * 70)
print(result)


ONE-DAY TRAVEL PLAN — BOSTON
CURRENT WEATHER

City: Boston  
Temperature: 15 °C  
Condition: Overcast clouds  

ONE-DAY ITINERARY  

**Morning**  
- Start at the **New England Aquarium**. Arrive early to explore the marine exhibits and catch the daily animal presentations.  

**Afternoon**  
- Walk the Freedom Trail to the historic **Franklin Park Zoo**. Enjoy the zoo’s diverse animal collection and the beautiful surrounding park.  

**Evening**  
- Visit the **Museum of Fine Arts, Boston** for late‑day gallery viewing and special exhibitions.  

ESTIMATED ATTRACTION COST  

1. New England Aquarium — $34.00 (Adult)  
2. Franklin Park Zoo — $27.95 (Adult)  
3. Museum of Fine Arts, Boston — $30.00 (Adult)  

Total estimated cost: **$91.95**  

TRAVEL TIP  
Buy tickets online in advance to skip lines, and use the “CharlieCard” for discounted public‑transport fares on the subway and bus network. Enjoy walking the Freedom Trail between attractions to soak up Boston’s rich history.

